# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassaan-Raza/FlyRank-Intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Method: Random Forest classifier, compared against Logistic Regression as a
simpler baseline model.

Why: this is a binary classification task feeding a ranked queue (same framing
as Week 2). Logistic Regression gives an interpretable linear baseline. Random
Forest can capture non-linear interactions between weak signals (impressions,
position, freshness, engagement) the way the Week 4 fixed rule couldn't,
matches the pattern FlyRank's own starter pipeline documented (baseline 0.240
Precision@50 -> Random Forest 0.740). Gradient Boosting was considered but
skipped this round, Random Forest already tests whether non-linearity helps
before reaching for something heavier.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

Group-based holdout by client_id, not a random row split. Rows from the same
client can share patterns (similar content strategy, similar baseline traffic
levels), so a random split risks leaking client-specific behavior between
train and test. 80/20 split, whole clients kept out of training entirely,
same validation logic FlyRank's own starter pipeline used.

Excluded from features (leakage risk): trend_pct (the near-certain source of
trend_direction itself), impressions_last_30d, impressions_prev_30d,
clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d,
these are the likely raw components the trend label was calculated from,
using them would let the model reconstruct the answer instead of finding
real signal.

In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

X = df[feature_cols].fillna(0)
y = df['is_declining']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx].copy()

print(f"Train: {len(X_train)} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"Test: {len(X_test)} rows, {groups.iloc[test_idx].nunique()} clients")

Train: 23837 rows, 25 clients
Test: 6163 rows, 7 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

# Baseline: recreate Week 4's fixed rule, scored on the SAME test set
df_test['baseline_score'] = 0
mask = (df_test['trend_direction'] == 'down') & (df_test['impressions_90d'] >= 100)
df_test.loc[mask, 'baseline_score'] = df_test.loc[mask, 'impressions_90d']

baseline_p50 = precision_at_k(y_test.reset_index(drop=True), df_test['baseline_score'].reset_index(drop=True), k=50)

# Logistic Regression
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
lr_p50 = precision_at_k(y_test.reset_index(drop=True), pd.Series(lr_probs), k=50)
lr_auc = roc_auc_score(y_test, lr_probs)
lr_ap = average_precision_score(y_test, lr_probs)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(y_test.reset_index(drop=True), pd.Series(rf_probs), k=50)
rf_auc = roc_auc_score(y_test, rf_probs)
rf_ap = average_precision_score(y_test, rf_probs)

comparison = pd.DataFrame({
    'Method': ['Baseline rule (Week 4)', 'Logistic Regression', 'Random Forest'],
    'Precision@50': [baseline_p50, lr_p50, rf_p50],
    'ROC AUC': [None, lr_auc, rf_auc],
    'Average Precision': [None, lr_ap, rf_ap]
})
print(comparison)
# What does the model find that the fixed rule completely misses?
rf_top50_idx = pd.Series(rf_probs).nlargest(50).index
rf_top50 = df_test.reset_index(drop=True).loc[rf_top50_idx]
new_candidates = rf_top50[rf_top50['baseline_score'] == 0]
print(f"\nOf RF's top 50, {len(new_candidates)} were scored 0 by the baseline rule entirely")
print(f"Of those, {new_candidates['is_declining'].sum()} are genuinely declining (true positives the rule structurally cannot find)")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                   Method  Precision@50   ROC AUC  Average Precision
0  Baseline rule (Week 4)          1.00       NaN                NaN
1     Logistic Regression          0.72  0.602502           0.600878
2           Random Forest          0.62  0.606583           0.599130

Of RF's top 50, 20 were scored 0 by the baseline rule entirely
Of those, 1 are genuinely declining (true positives the rule structurally cannot find)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top features by importance:")
print(importances.head(10))

# Look at false negatives: real declines the model ranked low
df_test_reset = df_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)
df_test_reset['rf_prob'] = rf_probs
false_negatives = df_test_reset[(y_test_reset == 1) & (df_test_reset['rf_prob'] < 0.3)]
print(f"\nFalse negatives (missed declines): {len(false_negatives)}")
print(false_negatives[['impressions_90d', 'avg_position', 'engagement_rate']].describe())

Top features by importance:
avg_position             0.121604
impressions_90d          0.118998
days_with_impressions    0.096665
content_age_days         0.089269
word_count               0.069070
char_count               0.065888
sessions_90d             0.055025
ctr                      0.052287
days_with_sessions       0.050995
scroll_rate              0.050102
dtype: float64

False negatives (missed declines): 220
       impressions_90d  avg_position  engagement_rate
count       220.000000    220.000000       220.000000
mean       1348.404545     20.271364         2.435227
std        3677.302977     17.064395         7.804334
min           1.000000      0.000000         0.000000
25%           3.000000      5.750000         0.000000
50%          20.500000     12.500000         0.000000
75%        1382.250000     33.800000         0.000000
max       27326.000000     76.000000        50.000000


**Model vs baseline:** Precision@50 is NOT a fair comparison here. The Week 4
baseline rule only assigns a nonzero score to rows where trend_direction ==
'down', which IS the label itself, so any row it ranks is guaranteed
positive by construction. Its 1.00 Precision@50 reflects this circularity,
not real predictive skill.

The fairer comparison is ROC AUC and Average Precision, computed across the
full ranking, not just the rule's self-selected top picks: Logistic
Regression (AUC 0.60, AP 0.60) and Random Forest (AUC 0.61, AP 0.60) both
land modestly above random chance (0.50), meaning there is real, if weak,
learnable signal in the observable features. Neither model shows a dramatic
edge yet, this is an honest, unglamorous result, not the sharp win the
starter pipeline's own documented numbers suggested.

**More importantly:** of RF's top 50, 20 were scored 0 by the baseline rule
entirely, and only 1 of those is genuinely declining. This is a real but
modest result, not a dramatic one: the model can surface candidates the
fixed rule structurally cannot see, but the hit rate on those novel
candidates (1 out of 20, 5%) is far lower than the rule's own gated
precision. That's an honest tradeoff to report: the rule is narrow but
accurate within its narrow gate, the model casts a wider net but dilutes
precision doing so. A useful next step would be raising the model's
confidence threshold on these "new" candidates specifically, rather than
treating its full top 50 as equally trustworthy.

**Top features:** avg_position and impressions_90d dominate, both directly
observable, sensible signals, not surprising given the baseline rule leaned
on similar logic.

**Errors:** false negatives (220 rows) skew toward low impressions (median 20.5)
and weaker position (median 12.5), the model is more confident on
high-visibility pages and misses quieter, lower-traffic declines, worth
flagging as a real limitation for a capstone review.

**Honest limitation:** trend_direction remains a current-window proxy label, not
a verified future outcome, this whole comparison describes ranking ability
on the present state, not forecasting skill.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.